<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/fundamentos/notebooks/c1_l2_retornos.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C1-L2 · Retornos simples vs logarítmicos
Calcula ambos sobre precios demo y verifica que los logs se suman. Solo librería estándar.

In [ ]:
import csv, math, urllib.request
from pathlib import Path

CSV = 'c1_l2_precios.csv'
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/fundamentos/data/' + CSV
csv_path = Path(CSV)
try:
    csv_path.write_bytes(urllib.request.urlopen(URL, timeout=15).read())
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data') / CSV, Path('data') / CSV, Path(CSV)]:
        if cand.exists():
            csv_path = cand
            break
closes = []
with open(csv_path, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        closes.append(float(row['close']))
print(f'días={len(closes)}  primero={closes[0]:.2f}  último={closes[-1]:.2f}')

In [ ]:
simples, logs = [], []
for p0, p1 in zip(closes[:-1], closes[1:]):
    simples.append((p1 - p0) / p0)
    logs.append(math.log(p1 / p0))
total_simple = (closes[-1] - closes[0]) / closes[0]
total_log = math.log(closes[-1] / closes[0])
print(f'retorno total simple: {total_simple:+.4%}')
print(f'suma de logs       : {sum(logs):+.6f}  vs log total {total_log:+.6f}')
print(f'producto de (1+r)  : {math.prod(1 + r for r in simples) - 1:+.4%} (debe igualar al total simple)')

## El ejemplo doloroso: +50% y −50%

In [ ]:
p = 100 * 1.5 * 0.5
print(f'100 → 150 → {p:.0f}: el promedio simple dice 0%, la cuenta dice {(p/100)-1:+.0%}')
r_log = math.log(1.5) + math.log(0.5)
print(f'en log: {math.log(1.5):+.4f} + {math.log(0.5):+.4f} = {r_log:+.4f} → exp = {math.exp(r_log):.2f}x (honesto)')

In [ ]:
# Chequeos automáticos
assert abs(sum(logs) - total_log) < 1e-9, 'los logs diarios deben sumar el log total'
assert abs((math.prod(1 + r for r in simples) - 1) - total_simple) < 1e-9
assert abs(math.log(1.01) - 0.00995) < 1e-4, 'en ±1% simple≈log'
print('OK: log suma, simple encadena')